In [1]:
import os
import sys

# ecCodes (cfgrib) finds its DLL through PATH, which only has the env's Library/bin when the env is activated
_lib_bin = os.path.join(sys.prefix, "Library", "bin")
if os.name == "nt" and _lib_bin.lower() not in os.environ["PATH"].lower():
    os.environ["PATH"] = _lib_bin + os.pathsep + os.environ["PATH"]

# PROJ_LIB points at PostGIS's old proj.db on this machine and breaks every CRS lookup
if "postgis" in os.environ.get("PROJ_LIB", "").lower():
    os.environ.pop("PROJ_LIB")

import base64
import bz2
import functools
import gzip
import json
import lzma
import re
import shutil
import tarfile
import time
import traceback
import uuid
import warnings
import zipfile
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import geoparquet_io as gpio
import numpy as np
import pandas as pd
import pdal
import pyarrow as pa
import pyarrow.parquet as pq
import pyogrio
import pyproj
import pystac
import rasterio
import rasterio.errors
import rasterio.shutil
import rioxarray  # noqa: F401  (registers the .rio accessor)
import xarray as xr

import shapely
from pdal.libpdalpython import infer_reader_driver
from pystac.extensions.projection import ProjectionExtension

In [2]:

DATA_DIR = Path("../data").resolve()
OUT_DIR = DATA_DIR / "cng"
WORK_DIR = OUT_DIR / "_work"   # zips are extracted here, temporary VRTs live here

warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)
# consolidated metadata (one request to list a store) is not in the v3 spec yet, but zarr-python, xarray and GDAL read it
warnings.filterwarnings("ignore", message="Consolidated metadata is currently not part")

ASSUME_CRS = None    # e.g. "EPSG:6668" -> stamped on inputs that carry no CRS (GSI DEM1A tiles, bare shapefiles)
UUID_COL = "uuid"
GEOPARQUET_VERSION = "2.0"
ZARR_FORMAT = 3      # conda GDAL 3.13 / xarray read it; 2 for zarr-python 2 era tools (rasterio's pip wheel
                     # bundles GDAL 3.10, which reads neither: no v3 'bytes' codec, no blosc)
OVERWRITE = True

In [3]:

VECTOR_EXT = {".geojson", ".geojsonl", ".geojsons", ".shp", ".gpkg", ".gml", ".kml", ".kmz", ".fgb", ".gpx",
              ".tab", ".mif", ".parquet", ".geoparquet"}    # .parquet only with GeoParquet "geo" metadata

RASTER_EXT = {".tif", ".tiff", ".jp2", ".img", ".asc", ".vrt", ".he5", ".bil", ".dem", ".ers"}

DATACUBE_EXT = {".nc", ".nc4", ".grib", ".grb", ".grib2"}

POINTCLOUD_EXT = {".las", ".laz", ".e57", ".bpf", ".sbet", ".csd", ".qi", ".fbi", ".slpk"}

TEXT_EXT = {".xyz"}                          # gridded -> GDAL XYZ raster, scattered X Y Z rows -> PDAL readers.text

MULTI_EXT = {".gpkg", ".mbtiles"}            # can hold vector layers and raster tiles side by side

TAR_EXT = (".tar", ".tar.gz", ".tgz", ".tar.bz2", ".tbz2", ".tar.xz", ".txz")

ARCHIVE_EXT = (".zip", ".7z", ".gz", ".bz2", ".xz") + TAR_EXT

SIDECAR_EXT = {".shx", ".dbf", ".prj", ".cpg", ".sbn", ".sbx", ".qix", ".qmd", ".ovr", ".aux", ".msk", ".tfw",
               ".tifw", ".jgw", ".pgw", ".wld", ".md5", ".html", ".htm", ".xsd", ".qml", ".sld",
               ".lyr", ".safe", ".log", ".ipynb", ".py", ".idx", ".gfs"}    # .idx: cfgrib's GRIB index, .gfs: GDAL's GML schema cache

# best first: when one folder has the same stem in several vector formats only the first one is converted
VECTOR_PRIORITY = [".parquet", ".geoparquet", ".gpkg", ".fgb", ".shp", ".geojson", ".gml", ".xml", ".kml"]

In [4]:
@dataclass
class Item:
    kind: str      # vector | raster | datacube | pointcloud | unsupported
    path: Path     # what GDAL/PDAL opens
    source: str    # the original file: the input file itself, or the archive it was extracted from
    reason: str = None    # why an unsupported item is skipped


_NUM = re.compile(r"[-+]?(\d+\.?\d*|\.\d+)([eE][-+]?\d+)?")
_TEXT_COLUMNS = {3: "X Y Z", 4: "X Y Z Intensity", 6: "X Y Z Red Green Blue", 7: "X Y Z Intensity Red Green Blue"}


def _text_reader(p):
    """readers.text options for an ASCII point cloud: a header naming X, Y, Z, or bare numeric rows
    (columns named from their count). None when the file is not one (readme, metadata, ...)."""
    try:
        with open(p, encoding="utf-8", errors="replace") as f:
            lines = [l.strip() for l in (next(f, "") for _ in range(50))]
    except OSError:
        return None
    skip = 0
    while skip < len(lines) and (not lines[skip] or lines[skip].startswith(("#", "//"))):
        skip += 1
    rows = [l for l in lines[skip:] if l]
    if len(rows) < 2:
        return None
    sep = next((s for s in (",", ";", "\t") if s in rows[1]), " ")
    split = lambda l: [t.strip('" ') for t in (l.split() if sep == " " else l.split(sep))]
    first, data = split(rows[0]), split(rows[1])
    if len(data) < 3 or not all(_NUM.fullmatch(t) for t in data):
        return None
    opts = dict(type="readers.text", separator=sep, skip=skip)
    if all(_NUM.fullmatch(t) for t in first):
        names = _TEXT_COLUMNS.get(len(first), "X Y Z " + " ".join(f"Extra{i}" for i in range(4, len(first) + 1)))
        return opts | dict(header=sep.join(names.split()))
    return opts if {"x", "y", "z"} <= {t.lower() for t in first} else None


def pc_reader(path):
    """pdal.Reader kwargs for where PDAL can't infer the reader from the file name."""
    path = Path(path)
    if path.is_dir():
        return dict(type="readers.tiledb")
    if path.suffix.lower() in TEXT_EXT:
        return _text_reader(path) or {}
    return {}


@functools.cache
def pc_info(path):
    """Point count, bounds and SRS WKT. quickinfo only reads the header; readers that have none
    (text, ptx, obj, ...) are streamed once instead."""
    reader = pdal.Reader(str(path), **pc_reader(path))
    try:
        info = next(iter(reader.pipeline().quickinfo.values()))
        b = info.get("bounds") or {}
        return dict(count=int(info.get("num_points", 0)), srs=(info.get("srs") or {}).get("wkt") or "",
                    bounds=[b.get("minx"), b.get("miny"), b.get("maxx"), b.get("maxy")] if b else None)
    except RuntimeError:    # "No summary data available"
        p = reader.pipeline()
        arrays = p.iterator(chunk_size=1_000_000) if p.streamable else (p.execute(), p.arrays)[1]
        count, lo, hi = 0, np.full(2, np.inf), np.full(2, -np.inf)
        for a in arrays:
            if len(a):
                xy = np.column_stack([a["X"], a["Y"]])
                count, lo, hi = count + len(a), np.minimum(lo, xy.min(0)), np.maximum(hi, xy.max(0))
        return dict(count=count, srs=p.srswkt2 or "", bounds=[*lo, *hi] if count else None)


def _probe_vector(p):
    try:
        return len(pyogrio.list_layers(p)) > 0
    except Exception:
        return False


def _probe_raster(p):
    try:
        with rasterio.open(p) as src:
            return bool(src.count or src.subdatasets)
    except Exception:
        return False


def _head(p, n=1024):
    try:
        with open(p, "rb") as f:
            return f.read(n)
    except OSError:
        return b""


def _is_grib(p):
    """GRIB messages start with 'GRIB', possibly after a WMO bulletin header line."""
    head = _head(p)
    return head.startswith(b"GRIB") or b"\nGRIB" in head


def _is_geoparquet(p):
    try:
        return b"geo" in (pq.read_schema(p).metadata or {})
    except Exception:
        return False


def _probe_pointcloud(p):
    try:
        return infer_reader_driver(str(p)) not in ("", "readers.gdal", "readers.text") and pc_info(p)["count"] > 0
    except Exception:
        return False


def classify(p):
    """(kinds, reason): kinds is one or more of vector / raster / pointcloud, or [] with the reason it is skipped
    (reason None for sidecars, which belong to a dataset next to them)."""
    name, ext = p.name.lower(), p.suffix.lower()
    if name == "ept.json":
        return ["pointcloud"], None
    if name.endswith((".aux.xml", ".shp.xml")) or name.startswith(("ks-meta", ".")) or ext in SIDECAR_EXT:
        return [], None    # sidecars and hidden files (.extracted marker, .DS_Store, ...)
    if ext in DATACUBE_EXT:
        return ["datacube"], None
    if ext in POINTCLOUD_EXT:
        return ["pointcloud"], None
    if ext in TEXT_EXT:
        if _probe_raster(p):    # GDAL XYZ only opens regularly gridded rows
            return ["raster"], None
        if _text_reader(p):
            return ["pointcloud"], None
        return [], "text file is neither a regular XYZ grid nor X Y Z point rows"
    if ext in MULTI_EXT:
        kinds = [k for k, ok in (("vector", _probe_vector(p)), ("raster", _probe_raster(p))) if ok]
        return kinds, None if kinds else "GDAL found no vector layers or raster tiles"
    if ext in (".parquet", ".geoparquet") and not _is_geoparquet(p):
        return [], "Parquet without GeoParquet metadata (plain table)"
    if ext in VECTOR_EXT:
        return ["vector"], None
    if ext in RASTER_EXT:
        return ["raster"], None
    # unknown extension (.xml GML, GRIB as .bin, extension-less files, ...): magic bytes, then GDAL, then PDAL
    if _is_grib(p) or _head(p, 4).startswith(b"CDF"):
        return ["datacube"], None
    for kind, probe in (("vector", _probe_vector), ("raster", _probe_raster), ("pointcloud", _probe_pointcloud)):
        if probe(p):
            return [kind], None
    return [], f"no GDAL or PDAL driver can read {ext or 'extension-less'} files like this one"


def _dedupe_vectors(files):
    """Within one folder keep a single format per stem (N03 ships .shp + .geojson + GML .xml of the same data)."""
    rank = {e: i for i, e in enumerate(VECTOR_PRIORITY)}
    best = {}
    for f in files:
        ext = f.suffix.lower()
        if f.is_file() and ext in rank:
            if f.stem not in best or rank[ext] < rank[best[f.stem].suffix.lower()]:
                best[f.stem] = f
    return [f for f in files if f.suffix.lower() not in rank or best.get(f.stem) == f]


def _safe_entry(p):
    """Sentinel-2 opens from its MTD_*.xml, Sentinel-1 from manifest.safe."""
    mtd = sorted(p.glob("MTD_MSI*.xml"))
    return mtd[0] if mtd else p / "manifest.safe"


def _folder_dataset(p):
    """(kind, path) when folder `p` is one dataset that must not be walked, else None."""
    name = p.name.lower()
    if name.endswith(".safe"):
        return "raster", _safe_entry(p)
    if name.endswith(".zarr"):
        return "datacube", p
    if (p / "hdr.adf").exists():     # ArcInfo binary grid
        return "raster", p
    if name.endswith(".gdb") or any((p / f).exists() for f in ("arc.adf", "pal.adf", "lab.adf")):  # FileGDB / ArcInfo coverage
        return "vector", p
    if (p / "ept.json").exists():
        return "pointcloud", p / "ept.json"
    if (p / "__schema").is_dir() or (p / "__array_schema.tdb").exists():    # TileDB array
        return "pointcloud", p
    return None


def _extract(path):
    name = path.name.lower()
    dest = WORK_DIR / path.name.replace(".", "_")    # never ends in .SAFE, so it is walked as a folder
    if not (dest / ".extracted").exists():
        if name.endswith(".7z"):
            import py7zr    # optional: pip install py7zr  (imported first so a missing module leaves nothing behind)
        print(f"extracting {path.name} ...")
        shutil.rmtree(dest, ignore_errors=True)
        dest.mkdir(parents=True)
        if name.endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                z.extractall(dest)
        elif name.endswith(TAR_EXT):
            with tarfile.open(path) as t:
                t.extractall(dest, filter="data")
        elif name.endswith(".7z"):
            with py7zr.SevenZipFile(path) as z:
                z.extractall(dest)
        else:    # single compressed file: dem.tif.gz -> dem.tif
            opener = {".gz": gzip.open, ".bz2": bz2.open, ".xz": lzma.open}[path.suffix.lower()]
            with opener(path) as fi, open(dest / path.stem, "wb") as fo:
                shutil.copyfileobj(fi, fo)
        (dest / ".extracted").touch()
    return dest


def _walk(p, source, items):
    name = p.name.lower()
    skip = lambda reason: items.append(Item("unsupported", p, source or str(p), reason))
    if p.is_dir():
        if (ds := _folder_dataset(p)):
            items.append(Item(ds[0], ds[1], source or str(p)))
            if name.endswith(".gdb") and _probe_raster(p):    # FileGDB raster datasets
                items.append(Item("raster", p, source or str(p)))
            return
        try:
            # an ArcInfo workspace's info/ folder only holds tables for the grids/coverages next to it
            children = [f for f in sorted(p.iterdir()) if not (f.name.lower() == "info" and (f / "arc.dir").exists())]
        except OSError as e:
            return skip(f"cannot list folder: {e}")
        for f in _dedupe_vectors(children):
            _walk(f, source, items)
    elif name.endswith(ARCHIVE_EXT):
        try:
            dest = _extract(p)
        except ModuleNotFoundError as e:
            return skip(f"{e} (pip install py7zr to read .7z)")
        except Exception as e:    # corrupt / truncated / password-protected archive, disk full, ...
            shutil.rmtree(WORK_DIR / p.name.replace(".", "_"), ignore_errors=True)
            return skip(f"cannot extract archive: {type(e).__name__}: {e}")
        _walk(dest, source or str(p), items)    # everything inside points back to the archive
    else:
        kinds, reason = classify(p)
        for kind in kinds:
            items.append(Item(kind, p, source or str(p)))
        if reason:
            skip(reason)


def discover(inputs):
    pc_info.cache_clear()
    items = []
    for inp in inputs if isinstance(inputs, (list, tuple)) else [inputs]:
        p = Path(inp)
        if p.exists():
            _walk(p, None, items)
        else:
            items.append(Item("unsupported", p, str(p), "input not found"))
    return items

*Vector*
- missing CRS → `ASSUME_CRS` if set, otherwise warning
- null / empty geometries → dropped
- invalid geometries → `make_valid(method="structure")`, geometry collections reduced to the layer's dimension
- mixed single/multi (Polygon + MultiPolygon) → promoted to multi so the column has one type
- polygon rings → oriented counter-clockwise (GeoParquet recommendation)
- geographic CRS with coordinates outside ±180/±90 → warning (wrong CRS or swapped axes)
- mixed-type / non-string column names → cast so Parquet can store them

*Raster*
- no CRS but a geotransform → `ASSUME_CRS` if set (same fix as the GSI DEM1A tiles in FINAL_SINK_PIPELINE), otherwise warning
- GCP-only georeferencing (Sentinel-1 GRD) → warped to the GCP CRS while writing the COG
- no georeferencing at all → skipped (not geospatial)
- float raster without nodata → nodata set to NaN

*Datacube* (netCDF / GRIB / Zarr)
- no spatial grid (x/y or lon/lat dimensions) → skipped (not geospatial); 2-D lat/lon (curvilinear) → written as-is with a warning
- no CRS → EPSG:4326 when the axes are plain longitude/latitude (GRIB, most CF files), else `ASSUME_CRS` if set, otherwise warning; stored as a CF `spatial_ref` grid mapping
- longitudes 0..360 → -180..180 (sorted)
- netCDF / GRIB storage encodings (chunk sizes, zlib, ...) → dropped, only dtype / fill / scale / time units kept

*Point cloud*
- missing SRS → `ASSUME_CRS` if set, otherwise warning; empty clouds are skipped

In [5]:
@dataclass
class Record:
    kind: str
    source: str
    input: str
    output: str = None
    format: str = None
    dataset_uuid: str = field(default_factory=lambda: str(uuid.uuid4()))
    status: str = "pending"      # ok | exists | skipped | failed
    crs: str = None
    bounds: list = None
    count: int = None            # features / pixels (w x h x bands) / points
    fixes: list = field(default_factory=list)
    warnings: list = field(default_factory=list)
    error: str = None
    seconds: float = None
    created: str = None


def _crs_str(crs):
    """EPSG code when the CRS has one (also for WKT that only matches an EPSG definition), otherwise WKT."""
    if not crs:
        return None
    crs = pyproj.CRS.from_user_input(crs if isinstance(crs, str) else crs.to_wkt())
    epsg = crs.to_epsg(min_confidence=70)
    return f"EPSG:{epsg}" if epsg else crs.to_wkt()


def _reduce_collection(g, dim):
    """Keep only the parts of a GeometryCollection with the layer's dimension (0 point, 1 line, 2 polygon)."""
    parts = [p for p in shapely.get_parts(g) if shapely.get_dimensions(p) == dim]
    if not parts:
        return None
    return shapely.union_all(parts) if dim == 2 else shapely.GeometryCollection(parts).normalize()


def check_fix_vector(gdf, rec):
    geom_col = gdf.geometry.name

    # CRS
    if gdf.crs is None:
        if ASSUME_CRS:
            gdf = gdf.set_crs(ASSUME_CRS)
            rec.fixes.append(f"no CRS -> assumed {ASSUME_CRS}")
        else:
            rec.warnings.append("no CRS (set ASSUME_CRS to stamp one)")

    # null / empty
    arr = np.asarray(gdf.geometry.array, dtype=object)
    bad = shapely.is_missing(arr) | shapely.is_empty(arr)
    if bad.any():
        gdf, arr = gdf[~bad].copy(), arr[~bad]
        rec.fixes.append(f"dropped {int(bad.sum())} null/empty geometries")

    # invalid -> make_valid, then collapse collections back to the layer's dimension
    dims = shapely.get_dimensions(arr)
    dim = int(np.bincount(dims[dims >= 0]).argmax()) if len(arr) else 2
    invalid = ~shapely.is_valid(arr)
    if invalid.any():
        arr[invalid] = shapely.make_valid(arr[invalid], method="structure", keep_collapsed=False)
        gc = shapely.get_type_id(arr) == 7
        if gc.any():
            arr[gc] = [_reduce_collection(g, dim) for g in arr[gc]]
        rec.fixes.append(f"repaired {int(invalid.sum())} invalid geometries")
        still_bad = shapely.is_missing(arr) | shapely.is_empty(arr)
        if still_bad.any():
            gdf, arr = gdf[~still_bad].copy(), arr[~still_bad]
            rec.fixes.append(f"dropped {int(still_bad.sum())} geometries that collapsed during repair")

    # mixed single/multi -> multi
    types = shapely.get_type_id(arr)
    for single, multi, ctor, label in [(0, 4, shapely.multipoints, "Point"),
                                       (1, 5, shapely.multilinestrings, "LineString"),
                                       (3, 6, shapely.multipolygons, "Polygon")]:
        m = types == single
        if m.any() and (types == multi).any():
            arr[m] = ctor(arr[m], indices=np.arange(m.sum()))
            rec.fixes.append(f"promoted {int(m.sum())} {label} to Multi{label}")

    # CCW exterior rings
    arr = shapely.orient_polygons(arr, exterior_cw=False)
    gdf[geom_col] = gpd.GeoSeries(arr, index=gdf.index, crs=gdf.crs)

    # coordinate sanity for geographic CRS
    if gdf.crs is not None and gdf.crs.is_geographic and len(gdf):
        minx, miny, maxx, maxy = gdf.total_bounds
        if minx < -180.001 or maxx > 180.001 or miny < -90.001 or maxy > 90.001:
            rec.warnings.append(f"coordinates {np.round(gdf.total_bounds, 2).tolist()} outside lon/lat range "
                                f"for {_crs_str(gdf.crs)} - wrong CRS or swapped axes?")

    # columns Parquet can't store as-is
    gdf.columns = [str(c) for c in gdf.columns]
    for c in gdf.columns:
        if c != geom_col and gdf[c].dtype == object:
            kind = pd.api.types.infer_dtype(gdf[c], skipna=True)
            if kind.startswith("mixed") or kind == "bytes":
                gdf[c] = gdf[c].map(lambda v: None if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))
                rec.fixes.append(f"column '{c}' had mixed types ({kind}) -> cast to string")

    return gdf.reset_index(drop=True)


def check_fix_raster(src, rec):
    """Inspect an open raster; return the edits to apply on the VRT (crs, nodata) and COG warp options."""
    edits, warp = {}, {}
    gcps, gcp_crs = src.gcps
    has_transform = src.transform != rasterio.Affine.identity()

    if not has_transform:
        if gcps and gcp_crs is not None:
            warp["TARGET_SRS"] = _crs_str(gcp_crs)
            rec.fixes.append(f"GCP-only georeferencing ({len(gcps)} GCPs) -> warped to {warp['TARGET_SRS']}")
        else:
            return None, None    # not georeferenced -> not geospatial
    elif src.crs is None or src.crs.to_wkt().startswith(("ENGCRS", "LOCAL_CS")):
        # ENGCRS "unnamed": GDAL's placeholder for a world file (.tfw) without a .prj - no real CRS
        if ASSUME_CRS:
            edits["crs"] = ASSUME_CRS
            rec.fixes.append(f"no CRS -> assumed {ASSUME_CRS}")
        else:
            rec.warnings.append("no CRS (set ASSUME_CRS to stamp one)")

    if src.nodata is None and np.issubdtype(np.dtype(src.dtypes[0]), np.floating):
        edits["nodata"] = float("nan")
        rec.fixes.append("float raster without nodata -> nodata=NaN")
    elif src.nodata is None and src.colorinterp[0].name not in ("palette", "red"):
        rec.warnings.append("no nodata value set")

    return edits, warp


def _spatial_dims(ds):
    """(x, y) grid dimension names: CF axis / standard_name first, then the usual names. (None, None) if absent."""
    def find(axis, std, names):
        for d in ds.dims:
            attrs = ds[d].attrs if d in ds.coords else {}
            if attrs.get("axis") == axis or attrs.get("standard_name") in std:
                return d
        return next((d for d in ds.dims if str(d).lower() in names), None)
    x = find("X", ("longitude", "projection_x_coordinate", "grid_longitude"), ("x", "lon", "longitude", "easting"))
    y = find("Y", ("latitude", "projection_y_coordinate", "grid_latitude"), ("y", "lat", "latitude", "northing"))
    return (x, y) if x and y else (None, None)


def _is_lonlat(ds, x, y):
    """Plain longitude / latitude axes (not a rotated-pole grid_longitude)."""
    if ds[x].attrs.get("standard_name") == "longitude" and ds[y].attrs.get("standard_name") == "latitude":
        return True
    if str(ds[x].attrs.get("units", "")).lower() in ("degrees_east", "degree_east"):
        return True
    return str(x).lower() in ("lon", "longitude") and str(y).lower() in ("lat", "latitude")


def check_fix_datacube(ds, rec):
    """Fixed Dataset (CRS as a CF grid mapping, longitudes in -180..180), or None when it has no spatial grid."""
    # netCDF / GRIB storage settings (chunk sizes, zlib, ...) don't apply to Zarr; keep what changes the values
    for v in ds.variables.values():
        v.encoding = {k: e for k, e in v.encoding.items()
                      if k in ("dtype", "_FillValue", "scale_factor", "add_offset", "units", "calendar", "grid_mapping",
                               "coordinates")}
    x, y = _spatial_dims(ds)
    if x is None:
        if any(str(c).lower() in ("lat", "latitude") and ds[c].ndim == 2 for c in ds.coords):
            rec.warnings.append("curvilinear grid (2-D lat/lon coordinates) - written as-is, no CRS grid mapping")
            return ds
        return None

    crs = ds.rio.set_spatial_dims(x_dim=x, y_dim=y).rio.crs
    if crs is None:
        if _is_lonlat(ds, x, y):
            crs = "EPSG:4326"
            rec.fixes.append("lon/lat axes without a grid mapping -> EPSG:4326")
        elif ASSUME_CRS:
            crs = ASSUME_CRS
            rec.fixes.append(f"no CRS -> assumed {ASSUME_CRS}")
        else:
            rec.warnings.append("no CRS (set ASSUME_CRS to stamp one)")

    # 0..360 longitudes (GRIB, many climate models) -> -180..180 so the grid lines up with everything else
    if crs is not None and pyproj.CRS.from_user_input(crs).is_geographic and float(ds[x].max()) > 180:
        ds = ds.assign_coords({x: (((ds[x] + 180) % 360) - 180).assign_attrs(ds[x].attrs)}).sortby(x)
        rec.fixes.append("longitudes 0..360 -> -180..180")

    ds = ds.rio.set_spatial_dims(x_dim=x, y_dim=y)
    if crs is not None:
        ds = ds.rio.write_crs(crs)    # CF grid mapping variable `spatial_ref`, read by GDAL / rioxarray / QGIS
    return ds


def check_pointcloud(path, rec):
    info = pc_info(path)
    srs = info["srs"]
    if "header" in (reader := pc_reader(path)):
        rec.fixes.append(f"headerless text -> assumed columns {reader['header']!r}")
    if not srs:
        if ASSUME_CRS:
            rec.fixes.append(f"no SRS -> assumed {ASSUME_CRS}")
        else:
            rec.warnings.append("no SRS (set ASSUME_CRS to stamp one)")
    rec.count = info["count"]
    rec.bounds = info["bounds"]
    rec.crs = _crs_str(srs) if srs else None
    return rec.count > 0, bool(srs)

- *Vector:* every feature gets a UUID in `UUID_COL`. If the column already exists (e.g. re-running on an output)
  valid unique values are kept and only missing / invalid / duplicate ones are regenerated, so IDs are stable across runs.
- *Raster / datacube / point cloud:* there are no features, so the dataset UUID is embedded in the file
  (`CNG_UUID` GDAL tag on the COG, `CNG_UUID` root attribute on the Zarr, a `CNG` VLR in the COPC).
- Every output also has `dataset_uuid` in the manifest.

In [6]:
UUID_RE = r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}"


def set_feature_uuids(gdf, rec):
    if UUID_COL in gdf.columns:
        s = gdf[UUID_COL].astype("string")
        valid = s.str.fullmatch(UUID_RE).fillna(False).astype(bool)
        regen = (~valid | s.duplicated(keep="first")).to_numpy()
        if regen.any():
            s[regen] = [str(uuid.uuid4()) for _ in range(int(regen.sum()))]
            rec.fixes.append(f"regenerated {int(regen.sum())} missing/invalid/duplicate {UUID_COL} values")
        gdf[UUID_COL] = s.astype(str)
    else:
        gdf.insert(0, UUID_COL, [str(uuid.uuid4()) for _ in range(len(gdf))])
    return gdf


def uuid_tags(rec):
    return {"CNG_UUID": rec.dataset_uuid, "CNG_SOURCE": Path(rec.input).name}


def uuid_vlr(rec):
    return [{"user_id": "CNG", "record_id": 1, "description": "dataset uuid",
             "data": base64.b64encode(rec.dataset_uuid.encode()).decode()}]

- `write_geoparquet` **2.0** stores geometry as the native Parquet `GEOMETRY` logical type (CRS as PROJJSON on the type) and
  every row group carries bbox statistics, so DuckDB can prune spatially without an extra column. **1.1** stores WKB
  plus a `bbox` covering column, for tools that can't read 2.0 yet.
- Rasters go source → VRT (tags/CRS/nodata edits, no pixel copy) → COG. 
- Datacubes (netCDF, GRIB, Zarr) open lazily with xarray (cfgrib for GRIB, which splits a file into one cube per
  level type / step), get their CRS through rioxarray and are written as Zarr with 512 × 512 chunks on the grid.
- Point clouds go through PDAL `writers.copc`.

In [7]:
def write_geoparquet(gdf, path):
    """
    Write `gdf` as a Hilbert-sorted, zstd-compressed GeoParquet file via
    geoparquet_io, in GEOPARQUET_VERSION. 2.0 needs no bbox column (row-group
    geo statistics do the pruning); 1.1 gets the bbox covering column.
    """
    pa_table = pa.table(gdf.to_arrow())
    table = gpio.Table(pa_table, crs=gdf.crs.to_json_dict() if gdf.crs else None)
    if GEOPARQUET_VERSION == "1.1":
        table = table.add_bbox()
    table.sort_hilbert().write(
        path,
        geoparquet_version=GEOPARQUET_VERSION,
        compression="zstd",
        row_group_size_mb=256,
        overwrite=True,
    )


def cog_options(src, warp):
    integer = np.issubdtype(np.dtype(src.dtypes[0]), np.integer)
    paletted = src.colorinterp[0].name == "palette"
    categorical = paletted or (integer and src.count == 1)
    opts = dict(
        COMPRESS="DEFLATE",
        PREDICTOR="YES",
        BLOCKSIZE=512,
        BIGTIFF="IF_SAFER",
        NUM_THREADS="ALL_CPUS",
        OVERVIEW_RESAMPLING="NEAREST" if categorical else "AVERAGE",
    )
    if warp:
        opts.update(warp, WARP_RESAMPLING="NEAREST" if categorical else "BILINEAR")
    return opts


def is_cog(path):
    with rasterio.open(path) as ds:
        return ds.tags(ns="IMAGE_STRUCTURE").get("LAYOUT") == "COG"


def write_zarr(ds, out):
    """512 x 512 chunks on the grid, dask's ~128 MB 'auto' along time / level / ...; written to a temporary store
    and swapped in, so a failed run never leaves a half-written Zarr behind."""
    spatial = _spatial_dims(ds)
    ds = ds.chunk({d: 512 if d in spatial else "auto" for d in ds.dims})
    tmp = out.with_name(out.name + ".tmp")
    shutil.rmtree(tmp, ignore_errors=True)
    ds.to_zarr(tmp, mode="w", zarr_format=ZARR_FORMAT, consolidated=True)
    shutil.rmtree(out, ignore_errors=True)
    os.replace(tmp, out)


def write_copc(path, out, rec, assume_srs):
    writer = dict(filename=str(out), forward="all", extra_dims="all", vlrs=uuid_vlr(rec))
    if assume_srs:
        writer["a_srs"] = ASSUME_CRS
    (pdal.Reader(str(path), **pc_reader(path)) | pdal.Writer.copc(**writer)).execute()

One function per input family runs check & fix → UUID → convert for each unit. Failures are caught per unit
(recorded with the traceback's last line). Unsupported inputs are printed with their reason and recorded as `skipped` (kind `unsupported`, reason in `warnings`).

In [8]:
_used_names = set()


def _out_path(kind, name, ext):
    name = re.sub(r"[^\w\-.]+", "_", name).strip("_") or "dataset"
    base, i = name, 1
    while (kind, name) in _used_names:
        i += 1
        name = f"{base}_{i}"
    _used_names.add((kind, name))
    d = OUT_DIR / kind
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{name}{ext}"


def _stem(p):
    n = p.name
    for suffix in (".copc.laz", ".SAFE", ".safe"):
        n = n.removesuffix(suffix)
    if p.name in ("manifest.safe",) or p.name.startswith("MTD_MSI"):
        n = p.parent.name.removesuffix(".SAFE")      # SAFE products are named after their folder
    if p.name.lower() == "ept.json":
        n = p.parent.name
    return Path(n).stem if "." in n else n


def _finish(rec, out, t0, status="ok"):
    rec.output, rec.status = str(out), status
    rec.seconds = round(time.time() - t0, 2)
    rec.created = datetime.now(timezone.utc).isoformat(timespec="seconds")
    return rec


def _fail(rec, t0):
    rec.status = "failed"
    rec.error = traceback.format_exc().strip().splitlines()[-1]
    rec.seconds = round(time.time() - t0, 2)
    print(f"  FAILED {rec.input}: {rec.error}")
    return rec


def _read_vector(path, layer):
    if path.suffix.lower() in (".parquet", ".geoparquet"):
        # drop a 1.1 bbox covering column: write_geoparquet re-adds it for 1.1, and in 2.0 it would be a stale attribute
        gdf = gpd.read_parquet(path)
        geo = json.loads(pq.read_schema(path).metadata.get(b"geo", b"{}"))
        covering = geo.get("columns", {}).get(geo.get("primary_column"), {}).get("covering", {}).get("bbox", {})
        return gdf.drop(columns=[c for c in {v[0] for v in covering.values()} if c in gdf.columns])
    opts = {}
    if path.suffix.lower() in (".csv", ".tsv"):
        opts = dict(X_POSSIBLE_NAMES="lon*,lng,x,longitude,経度", Y_POSSIBLE_NAMES="lat*,y,latitude,緯度",
                    GEOM_POSSIBLE_NAMES="geom*,wkt,the_geom", KEEP_GEOM_COLUMNS="NO", AUTODETECT_TYPE="YES")
    try:
        return pyogrio.read_dataframe(path, layer=layer, use_arrow=True, **opts)
    except UnicodeDecodeError:
        # Japanese government data (CSV / shapefiles without .cpg) is often Shift-JIS
        return pyogrio.read_dataframe(path, layer=layer, encoding="cp932", **opts)


def convert_vector(item):
    layers = [None] if item.path.suffix.lower() in (".parquet", ".geoparquet") else \
        [l[0] for l in pyogrio.list_layers(item.path)]
    records = []
    for layer in layers:
        t0 = time.time()
        rec = Record("vector", item.source, str(item.path) + (f"::{layer}" if len(layers) > 1 else ""))
        out = _out_path("vector", _stem(item.path) + (f"__{layer}" if len(layers) > 1 else ""), ".parquet")
        rec.format = f"GeoParquet {GEOPARQUET_VERSION}"
        if out.exists() and not OVERWRITE:
            records.append(_finish(rec, out, t0, "exists"))
            continue
        try:
            gdf = _read_vector(item.path, layer)
            if not isinstance(gdf, gpd.GeoDataFrame) or gdf.geometry is None:
                rec.status, rec.warnings = "skipped", ["no geometry column (non-spatial table)"]
                records.append(rec)
                continue
            if gdf.crs is None and item.path.suffix.lower() in (".csv", ".tsv") and not ASSUME_CRS:
                gdf = gdf.set_crs(4326)               # lon/lat columns are WGS84 unless told otherwise
                rec.fixes.append("CSV lon/lat columns -> assumed EPSG:4326")
            gdf = check_fix_vector(gdf, rec)          # 2. check & fix
            if gdf.empty:
                rec.status = "skipped"
                rec.warnings.append("no features left after checks")
                records.append(rec)
                continue
            gdf = set_feature_uuids(gdf, rec)         # 3. uuid
            write_geoparquet(gdf, out)                # 4. convert
            rec.crs, rec.count = _crs_str(gdf.crs), len(gdf)
            rec.bounds = [round(float(v), 6) for v in gdf.total_bounds]
            records.append(_finish(rec, out, t0))
        except Exception:
            records.append(_fail(rec, t0))
    return records


def _subdataset_name(sd):
    parts = [p.strip('"') for p in sd.split(":")[1:]]
    return "_".join(p for p in parts if len(p) > 1 and not re.search(r"[\\/]", p))


def convert_raster(item):
    with rasterio.open(item.path) as src:
        targets = [(sd, _subdataset_name(sd)) for sd in src.subdatasets] if (src.count == 0 or
                   src.driver == "SENTINEL2") else [(str(item.path), None)]
    records = []
    for target, sub in targets:
        t0 = time.time()
        rec = Record("raster", item.source, target, format="COG")
        out = _out_path("raster", _stem(item.path) + (f"__{sub}" if sub else ""), ".tif")
        if out.exists() and not OVERWRITE:
            records.append(_finish(rec, out, t0, "exists"))
            continue
        vrt = WORK_DIR / (out.stem + ".vrt")
        try:
            with rasterio.open(target) as src:
                edits, warp = check_fix_raster(src, rec)          # 2. check & fix
                if edits is None:
                    rec.status = "skipped"
                    rec.warnings.append("no georeferencing (no CRS/geotransform/GCPs) - not geospatial")
                    records.append(rec)
                    continue
                opts = cog_options(src, warp)
            WORK_DIR.mkdir(parents=True, exist_ok=True)
            rasterio.shutil.copy(target, vrt, driver="VRT")
            with rasterio.open(vrt, "r+") as v:                    # edits land in the VRT, no pixel copy
                v.update_tags(**uuid_tags(rec))                    # 3. uuid
                if "crs" in edits:
                    v.crs = rasterio.crs.CRS.from_user_input(edits["crs"])
                if "nodata" in edits:
                    v.nodata = edits["nodata"]
            tmp = out.with_suffix(".tmp.tif")
            rasterio.shutil.copy(vrt, tmp, driver="COG", **opts)  # 4. convert
            os.replace(tmp, out)
            with rasterio.open(out) as ds:
                rec.crs, rec.count = _crs_str(ds.crs), ds.width * ds.height * ds.count
                rec.bounds = [round(float(v), 6) for v in ds.bounds]
                if ds.tags().get("CNG_UUID") != rec.dataset_uuid:
                    rec.warnings.append("CNG_UUID tag missing from output")
            if not is_cog(out):
                rec.warnings.append("output does not report LAYOUT=COG")
            records.append(_finish(rec, out, t0))
        except Exception:
            records.append(_fail(rec, t0))
        finally:
            vrt.unlink(missing_ok=True)
    return records


def _open_cubes(path):
    """[(name suffix, Dataset)], lazily opened. A GRIB file mixing level types / steps is several hypercubes,
    cfgrib returns one Dataset per hypercube. decode_coords="all" makes the CF grid mapping variable a coordinate,
    which is where rioxarray reads the CRS from."""
    path = Path(path)
    if path.is_dir():
        return [(None, xr.open_zarr(path, decode_coords="all"))]
    if _is_grib(path):
        import cfgrib
        cubes = cfgrib.open_datasets(str(path), backend_kwargs={"indexpath": ""}, chunks={})
        if len(cubes) == 1:
            return [(None, cubes[0])]
        level = lambda ds: next((v.attrs.get("GRIB_typeOfLevel") for v in ds.data_vars.values()), None)
        return [(level(ds) or str(i), ds) for i, ds in enumerate(cubes)]
    return [(None, xr.open_dataset(path, chunks={}, decode_coords="all"))]


def convert_datacube(item):
    records = []
    for sub, cube in _open_cubes(item.path):
        t0 = time.time()
        rec = Record("datacube", item.source, str(item.path) + (f"::{sub}" if sub else ""), format=f"Zarr v{ZARR_FORMAT}")
        out = _out_path("datacube", _stem(item.path) + (f"__{sub}" if sub else ""), ".zarr")
        try:
            if out.exists() and not OVERWRITE:
                records.append(_finish(rec, out, t0, "exists"))
                continue
            ds = check_fix_datacube(cube, rec)                   # 2. check & fix
            if ds is None:
                rec.status = "skipped"
                rec.warnings.append("no spatial grid (x/y or lon/lat dimensions) - not geospatial")
                records.append(rec)
                continue
            ds.attrs.update(uuid_tags(rec))                      # 3. uuid
            write_zarr(ds, out)                                  # 4. convert
            with xr.open_zarr(out, decode_coords="all") as z:
                rec.crs = _crs_str(z.rio.crs)
                rec.count = sum(int(v.size) for v in z.data_vars.values())
                try:
                    rec.bounds = [round(float(b), 6) for b in z.rio.set_spatial_dims(*_spatial_dims(z)).rio.bounds()]
                except Exception:    # curvilinear / no grid mapping
                    pass
                if z.attrs.get("CNG_UUID") != rec.dataset_uuid:
                    rec.warnings.append("CNG_UUID attribute missing from output")
            records.append(_finish(rec, out, t0))
        except Exception:
            records.append(_fail(rec, t0))
        finally:
            cube.close()
    return records


def convert_pointcloud(item):
    t0 = time.time()
    rec = Record("pointcloud", item.source, str(item.path), format="COPC")
    out = _out_path("pointcloud", _stem(item.path), ".copc.laz")
    if out.exists() and not OVERWRITE:
        return [_finish(rec, out, t0, "exists")]
    try:
        has_points, has_srs = check_pointcloud(item.path, rec)   # 2. check
        if not has_points:
            rec.status = "skipped"
            rec.warnings.append("no points")
            return [rec]
        write_copc(item.path, out, rec, assume_srs=not has_srs and bool(ASSUME_CRS))   # 3. uuid + 4. convert
        return [_finish(rec, out, t0)]
    except Exception:
        return [_fail(rec, t0)]


CONVERTERS = {"vector": convert_vector, "raster": convert_raster, "datacube": convert_datacube,
              "pointcloud": convert_pointcloud}


def write_manifest(records):
    """Merge this run into OUT_DIR/manifest.json, keyed by output path (re-runs replace their own entries)."""
    path = OUT_DIR / "manifest.json"
    old = json.loads(path.read_text(encoding="utf-8")) if path.exists() else []
    new = [asdict(r) for r in records if r.status != "exists"]
    keys = {r["output"] or r["input"] for r in new}
    merged = [r for r in old if (r["output"] or r["input"]) not in keys] + new
    path.write_text(json.dumps(merged, indent=2, ensure_ascii=False), encoding="utf-8")
    write_file_index(merged)
    return path


INDEX_COLS = ["source", "input", "output", "dataset_uuid", "kind", "format", "status", "created"]


def write_file_index(manifest):
    """OUT_DIR/file_index.csv: input file -> output file -> dataset UUID, one row per converted output
    (skipped / unsupported / failed entries stay in the manifest only).
    utf-8-sig so Excel shows the Japanese file names correctly."""
    df = pd.DataFrame([r for r in manifest if r["status"] == "ok"], columns=INDEX_COLS).sort_values(["kind", "output"], na_position="last")
    df.to_csv(OUT_DIR / "file_index.csv", index=False, encoding="utf-8-sig")


def _stac_path(item, inputs):
    """Where an output sits in this run's STAC tree: its file path relative to the folder holding the run input
    it came from. An archive stands in for the folder it was extracted to; SAFE / EPT entry files for their folder."""
    src = Path(item.source)
    path = src
    dest = WORK_DIR / src.name.replace(".", "_")
    if item.path.is_relative_to(dest):
        inner = item.path
        if inner.name.lower() in ("manifest.safe", "ept.json") or inner.name.startswith("MTD_MSI"):
            inner = inner.parent
        path = src / inner.relative_to(dest)
    top = next((Path(i) for i in inputs if src == Path(i) or Path(i) in src.parents), src)
    return path.relative_to(top.parent).as_posix()


def run_pipeline(inputs):
    inputs = list(inputs) if isinstance(inputs, (list, tuple)) else [inputs]
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    _used_names.clear()
    items = discover(inputs)                                       # 1. input
    print(f"found {len(items)} datasets: " + ", ".join(f"{k}={sum(i.kind == k for i in items)}"
                                                       for k in [*CONVERTERS, "unsupported"]))
    records, stac_paths = [], []
    for item in items:
        n = len(records)
        if item.kind == "unsupported":
            print(f"[skipped] {item.path.name}: {item.reason}")
            records.append(Record("unsupported", item.source, str(item.path), status="skipped", warnings=[item.reason]))
        else:
            print(f"[{item.kind}] {item.path.name}")
            t0 = time.time()
            try:
                records += CONVERTERS[item.kind](item)
            except Exception:    # e.g. the file can't even be opened to list its layers / subdatasets
                records.append(_fail(Record(item.kind, item.source, str(item.path)), t0))
        stac_paths += [_stac_path(item, inputs)] * (len(records) - n)
    manifest = write_manifest(records)
    print(f"manifest: {manifest}")
    return pd.DataFrame([asdict(r) for r in records]).assign(stac_path=stac_paths)

In [9]:
inputs = [
    # --- formats in the lists: expected ok
    DATA_DIR / "meiji_jingu_boundary.geojson",
    DATA_DIR / "chiba_city_admin.geoparquet",
    DATA_DIR / "_flood_serve/chiba info all/cloud version/chiba_roads_by_city/勝浦市_roads.parquet",
    DATA_DIR / "gm-jpn-all_u_2_2/gm-jpn-all_u_2_2/airp_jpn.shp",                  
    DATA_DIR / "city_1km_buffer.zip",                                               
    DATA_DIR / "FG-GML-533945-DEM1A-20250822.zip",                                  
    DATA_DIR / "gm-jpn-lc_u_1_1/jpn/lc.tif",                                        
    DATA_DIR / "cloud_dem_test/tokyo_sample_sink.tif",
    DATA_DIR / "chiba_city.tif",                                                    
    DATA_DIR / "class_3.las",
    DATA_DIR / "class_3.copc.laz",

    
    DATA_DIR / "data.csv",                                                         
    DATA_DIR / "output/summary_report.xlsx",
    DATA_DIR / "chiba info all/chiba_stac/buildings/chiba-一宮町/chiba-一宮町.json",  
    DATA_DIR / "gunma info all/gunma_checkpoints/tile_dem_source.json",             
    DATA_DIR / "lst_heatmap.png",                                                   
    DATA_DIR / "chiba info all/flood_sink_comparison/oami_native_preview.png",
    DATA_DIR / "gm-jpn-lc_u_1_1/jpn/Read_me.txt",
    DATA_DIR / "chiba info all/chiba_stac_processed_at.txt",
    DATA_DIR / "cloud_dem_test/t5.pdf",                                             
    DATA_DIR / "cloud_dem_test/t4.svg",                                             
    DATA_DIR / "gm-jpn-all_u_2_2/gm-jpn-all_u_2_2/bnd_jpn.met",                     

    DATA_DIR / "09LD1743",                                                          
    DATA_DIR / "does_not_exist.tif",
    DATA_DIR / "dummy_datacube.nc",                
    DATA_DIR / "NC_H08_20161117_0230_JP02.zip",                                
]
results = run_pipeline(inputs)
results[["status", "kind", "input", "output", "crs", "count", "fixes", "warnings", "error", "seconds"]]

found 139 datasets: vector=8, raster=5, datacube=17, pointcloud=2, unsupported=107
[vector] meiji_jingu_boundary.geojson
[vector] chiba_city_admin.geoparquet
[vector] 勝浦市_roads.parquet
[vector] airp_jpn.shp
[vector] city_1km_buffer.shp
[skipped] FG-GML-5339-45-00-DEM1A-20250822.xml: no GDAL or PDAL driver can read .xml files like this one
[skipped] FG-GML-5339-45-01-DEM1A-20250822.xml: no GDAL or PDAL driver can read .xml files like this one
[skipped] FG-GML-5339-45-02-DEM1A-20250822.xml: no GDAL or PDAL driver can read .xml files like this one
[skipped] FG-GML-5339-45-03-DEM1A-20250822.xml: no GDAL or PDAL driver can read .xml files like this one
[skipped] FG-GML-5339-45-04-DEM1A-20250822.xml: no GDAL or PDAL driver can read .xml files like this one
[skipped] FG-GML-5339-45-05-DEM1A-20250822.xml: no GDAL or PDAL driver can read .xml files like this one
[skipped] FG-GML-5339-45-06-DEM1A-20250822.xml: no GDAL or PDAL driver can read .xml files like this one
[skipped] FG-GML-5339-45-07-D

,status,kind,input,output,crs,count,fixes,warnings,error,seconds
0,ok,vector,C:\Users\Pedro\Documents\PSS\data\meiji_jingu_...,C:\Users\Pedro\Documents\PSS\data\cng\vector\m...,EPSG:4326,1.0,[],[],None,1.32
1,ok,vector,C:\Users\Pedro\Documents\PSS\data\chiba_city_a...,C:\Users\Pedro\Documents\PSS\data\cng\vector\c...,EPSG:6668,3.0,[],[],None,0.29
2,ok,vector,C:\Users\Pedro\Documents\PSS\data\_flood_serve...,C:\Users\Pedro\Documents\PSS\data\cng\vector\勝...,EPSG:6668,275.0,[],[],None,0.28
3,ok,vector,C:\Users\Pedro\Documents\PSS\data\gm-jpn-all_u...,C:\Users\Pedro\Documents\PSS\data\cng\vector\a...,EPSG:4612,92.0,[],[],None,0.28
4,ok,vector,C:\Users\Pedro\Documents\PSS\data\cng\_work\ci...,C:\Users\Pedro\Documents\PSS\data\cng\vector\c...,EPSG:3857,1.0,[],[],None,0.32
...,...,...,...,...,...,...,...,...,...,...
136,ok,datacube,C:\Users\Pedro\Documents\PSS\data\cng\_work\NC...,C:\Users\Pedro\Documents\PSS\data\cng\datacube...,EPSG:4326,2230503.0,[lon/lat axes without a grid mapping -> EPSG:4...,[],None,0.20
137,ok,datacube,C:\Users\Pedro\Documents\PSS\data\cng\_work\NC...,C:\Users\Pedro\Documents\PSS\data\cng\datacube...,EPSG:4326,2230503.0,[lon/lat axes without a grid mapping -> EPSG:4...,[],None,0.19
138,ok,datacube,C:\Users\Pedro\Documents\PSS\data\cng\_work\NC...,C:\Users\Pedro\Documents\PSS\data\cng\datacube...,EPSG:4326,2230503.0,[lon/lat axes without a grid mapping -> EPSG:4...,[],None,0.18
139,ok,datacube,C:\Users\Pedro\Documents\PSS\data\cng\_work\NC...,C:\Users\Pedro\Documents\PSS\data\cng\datacube...,EPSG:4326,2230503.0,[lon/lat axes without a grid mapping -> EPSG:4...,[],None,0.18


In [10]:
# results = run_pipeline([DATA_DIR])   # whole data folder

In [11]:
def validate_output(rec):
    path = Path(rec["output"])
    if rec["kind"] == "vector":
        md = pq.read_metadata(path)
        geo = json.loads(md.metadata[b"geo"])
        if geo["version"].startswith("2"):
            logical = str(md.schema.column(md.schema.names.index(geo["primary_column"])).logical_type)
            return logical.startswith("Geometry"), f"GeoParquet {geo['version']}, native type: {logical[:9]}"
        col = geo["columns"][geo["primary_column"]]
        return "covering" in col, f"GeoParquet {geo['version']}, bbox covering: {'covering' in col}"
    if rec["kind"] == "raster":
        with rasterio.open(path) as ds:
            ok = ds.tags(ns="IMAGE_STRUCTURE").get("LAYOUT") == "COG" and ds.tags().get("CNG_UUID") == rec["dataset_uuid"]
            return ok, f"LAYOUT={ds.tags(ns='IMAGE_STRUCTURE').get('LAYOUT')}, overviews={ds.overviews(1)}"
    if rec["kind"] == "datacube":
        with xr.open_zarr(path, decode_coords="all") as z:
            crs = _crs_str(z.rio.crs)
            return z.attrs.get("CNG_UUID") == rec["dataset_uuid"] and crs is not None, f"dims={dict(z.sizes)}, crs={crs}"
    if rec["kind"] == "pointcloud":
        info = pdal.Reader(str(path)).pipeline().quickinfo
        return "readers.copc" in info, f"read by {next(iter(info))}"


manifest = pd.read_json(OUT_DIR / "manifest.json")
done = manifest[manifest.status == "ok"]
checks = [validate_output(r) for r in done.to_dict("records")]
done.assign(valid=[c[0] for c in checks], detail=[c[1] for c in checks])[["kind", "output", "dataset_uuid", "valid", "detail"]]

,kind,output,dataset_uuid,valid,detail
101,vector,C:\Users\Pedro\Documents\PSS\data\cng\vector\m...,ae59ee93-9fc0-4448-9bcb-37ad435bc157,True,"GeoParquet 2.0.0, native type: Geometry("
102,vector,C:\Users\Pedro\Documents\PSS\data\cng\vector\c...,362f8d68-9445-425d-ad27-38460b526e03,True,"GeoParquet 2.0.0, native type: Geometry("
103,vector,C:\Users\Pedro\Documents\PSS\data\cng\vector\勝...,d3bcbd18-4b4c-4536-878a-f7b369009bc0,True,"GeoParquet 2.0.0, native type: Geometry("
104,vector,C:\Users\Pedro\Documents\PSS\data\cng\vector\a...,2c8a27ed-d35d-4c30-9220-7e24d4217b7a,True,"GeoParquet 2.0.0, native type: Geometry("
105,vector,C:\Users\Pedro\Documents\PSS\data\cng\vector\c...,88186ed7-af98-4dbc-8048-2acb23dc9b20,True,"GeoParquet 2.0.0, native type: Geometry("
206,raster,C:\Users\Pedro\Documents\PSS\data\cng\raster\l...,70ed7ca3-546f-46b8-a72c-2d3a43fc30e4,True,"LAYOUT=COG, overviews=[2, 4, 8, 16]"
207,raster,C:\Users\Pedro\Documents\PSS\data\cng\raster\t...,e489674b-fa28-45f2-bc5e-c129d1d49528,True,"LAYOUT=COG, overviews=[8, 2]"
208,raster,C:\Users\Pedro\Documents\PSS\data\cng\raster\c...,780fa156-6a8a-4ee7-bc07-d1442ffd85fe,True,"LAYOUT=COG, overviews=[]"
209,pointcloud,C:\Users\Pedro\Documents\PSS\data\cng\pointclo...,da9e52c4-1bb0-4962-8032-18d73588ef0c,True,read by readers.copc
210,pointcloud,C:\Users\Pedro\Documents\PSS\data\cng\pointclo...,8dbe6ff9-b50c-4e78-926d-8850d0c909e7,True,read by readers.copc


In [12]:
pd.read_csv(OUT_DIR / "file_index.csv")

,source,input,output,dataset_uuid,kind,format,status,created
0,C:\Users\Pedro\Documents\PSS\data\NC_H08_20161...,C:\Users\Pedro\Documents\PSS\data\cng\_work\NC...,C:\Users\Pedro\Documents\PSS\data\cng\datacube...,42cce279-7d55-46d5-a05b-d16b12a19380,datacube,Zarr v3,ok,2026-10-21T05:10:16+00:00
1,C:\Users\Pedro\Documents\PSS\data\NC_H08_20161...,C:\Users\Pedro\Documents\PSS\data\cng\_work\NC...,C:\Users\Pedro\Documents\PSS\data\cng\datacube...,047adaa8-969c-45ed-b3ea-4e7ccdd27cc4,datacube,Zarr v3,ok,2026-10-21T05:10:16+00:00
2,C:\Users\Pedro\Documents\PSS\data\NC_H08_20161...,C:\Users\Pedro\Documents\PSS\data\cng\_work\NC...,C:\Users\Pedro\Documents\PSS\data\cng\datacube...,6362bc06-226c-4426-a218-f0916557c6ec,datacube,Zarr v3,ok,2026-10-21T05:10:17+00:00
3,C:\Users\Pedro\Documents\PSS\data\NC_H08_20161...,C:\Users\Pedro\Documents\PSS\data\cng\_work\NC...,C:\Users\Pedro\Documents\PSS\data\cng\datacube...,763c1fef-f0f0-4788-a2a7-cb1b7b0d6f88,datacube,Zarr v3,ok,2026-10-21T05:10:18+00:00
4,C:\Users\Pedro\Documents\PSS\data\NC_H08_20161...,C:\Users\Pedro\Documents\PSS\data\cng\_work\NC...,C:\Users\Pedro\Documents\PSS\data\cng\datacube...,192c35e5-81ef-4545-bbe0-7ad1fa8d6232,datacube,Zarr v3,ok,2026-10-21T05:10:18+00:00
5,C:\Users\Pedro\Documents\PSS\data\NC_H08_20161...,C:\Users\Pedro\Documents\PSS\data\cng\_work\NC...,C:\Users\Pedro\Documents\PSS\data\cng\datacube...,8c8934d3-a3a2-4ef1-90b9-aed165de8978,datacube,Zarr v3,ok,2026-10-21T05:10:18+00:00
6,C:\Users\Pedro\Documents\PSS\data\NC_H08_20161...,C:\Users\Pedro\Documents\PSS\data\cng\_work\NC...,C:\Users\Pedro\Documents\PSS\data\cng\datacube...,063617f6-8bed-43ec-8fba-9a9be020e4b7,datacube,Zarr v3,ok,2026-10-21T05:10:18+00:00
7,C:\Users\Pedro\Documents\PSS\data\NC_H08_20161...,C:\Users\Pedro\Documents\PSS\data\cng\_work\NC...,C:\Users\Pedro\Documents\PSS\data\cng\datacube...,28bb6f0e-fe31-4b17-83ab-4a718f572948,datacube,Zarr v3,ok,2026-10-21T05:10:18+00:00
8,C:\Users\Pedro\Documents\PSS\data\NC_H08_20161...,C:\Users\Pedro\Documents\PSS\data\cng\_work\NC...,C:\Users\Pedro\Documents\PSS\data\cng\datacube...,feac160c-c8fe-4fd1-9f38-2f5f1bced402,datacube,Zarr v3,ok,2026-10-21T05:10:19+00:00
9,C:\Users\Pedro\Documents\PSS\data\NC_H08_20161...,C:\Users\Pedro\Documents\PSS\data\cng\_work\NC...,C:\Users\Pedro\Documents\PSS\data\cng\datacube...,3d697034-a967-439c-a45f-03c467087db6,datacube,Zarr v3,ok,2026-10-21T05:10:19+00:00


STAC catalog of the outputs of **this run** (`results`, status `ok`), shaped after the input folders:
- one input folder holding only files → a **Collection**, one Item per output file (each Item has one `data` asset)
- one input folder holding subfolders → a **Catalog**, one **Collection** per subfolder (deeper folders are flattened into it);
  files lying next to the subfolders become Items directly on the Catalog
- several inputs, or a single file → a `run_<timestamp>` Catalog: file inputs become its Items, folder inputs follow the rules above
  (when everything is loose files the root is a Collection)
- archives count as folders, unless they hold a single file

Written to `cng/stac/<folder name or run_timestamp>/`; earlier runs' catalogs are left alone.

In [13]:
STAC_DIR = OUT_DIR / "stac"
STAC_MEDIA = {"vector": pystac.MediaType.PARQUET, "raster": pystac.MediaType.COG,
              "datacube": pystac.MediaType.ZARR, "pointcloud": pystac.MediaType.COPC}
STAC_DESCRIPTION = {"vector": "GeoParquet vector layers", "raster": "Cloud-optimized GeoTIFFs",
                    "datacube": "Zarr datacubes", "pointcloud": "COPC point clouds"}


def bbox_wgs84(bounds, crs):
    """Native bounds -> [minx, miny, maxx, maxy] in EPSG:4326 (densified, so curved edges stay inside).
    None when there is no CRS or it can't be reprojected (local engineering CRS)."""
    if not bounds or not crs:
        return None
    try:
        t = pyproj.Transformer.from_crs(pyproj.CRS.from_user_input(crs), "EPSG:4326", always_xy=True)
        bbox = t.transform_bounds(*bounds, densify_pts=21)
    except pyproj.exceptions.ProjError:
        return None
    return [round(v, 7) for v in bbox] if all(np.isfinite(bbox)) else None


def stac_item(rec):
    out = Path(rec["output"])
    stem = out.name.removesuffix(".copc.laz") if out.name.endswith(".copc.laz") else out.stem
    bbox = bbox_wgs84(rec["bounds"], rec["crs"])
    item = pystac.Item(
        id=stem,                                  # unique within a kind: _out_path dedupes names per kind
        geometry=shapely.geometry.mapping(shapely.box(*bbox)) if bbox else None,
        bbox=bbox,
        datetime=datetime.fromisoformat(rec["created"]),
        properties={"title": stem, "dataset_uuid": rec["dataset_uuid"], "source": Path(rec["source"]).name,
                    "cng:format": rec["format"], "cng:count": rec["count"]},
    )
    if rec["crs"]:
        proj = ProjectionExtension.ext(item, add_if_missing=True)
        if rec["crs"].startswith("EPSG:"):
            proj.code = rec["crs"]
        else:
            proj.wkt2 = rec["crs"]
        proj.bbox = rec["bounds"]
    item.add_asset("data", pystac.Asset(href=str(out), media_type=STAC_MEDIA[rec["kind"]], roles=["data"],
                                        title=rec["format"]))
    return item


def _unique(name, used):
    base, i = name, 1
    while name in used:
        i += 1
        name = f"{base}_{i}"
    used.add(name)
    return name


def _tree(recs):
    """{"dirs": {name: node}, "files": {name: [records]}} from the records' stac_path."""
    root = {"dirs": {}, "files": {}}
    for r in recs:
        *dirs, name = Path(r["stac_path"]).parts
        node = root
        for d in dirs:
            node = node["dirs"].setdefault(d, {"dirs": {}, "files": {}})
        node["files"].setdefault(name, []).append(r)
    return root


def _collapse_archives(node):
    """An archive holding a single file is that file, not a folder."""
    for name, child in list(node["dirs"].items()):
        _collapse_archives(child)
        if name.lower().endswith(ARCHIVE_EXT) and not child["dirs"] and len(child["files"]) == 1:
            del node["dirs"][name]
            node["files"][name] = next(iter(child["files"].values()))
    return node


def _records(node):
    return [r for rs in node["files"].values() for r in rs] + [r for d in node["dirs"].values() for r in _records(d)]


def _items(recs):
    used, items = set(), []
    for r in recs:
        item = stac_item(r)
        item.id = _unique(item.id if item.id not in used else f"{item.id}_{r['kind']}", used)
        items.append(item)
    return items


def _collection(name, recs):
    items = _items(recs)
    no_bbox = [i.id for i in items if i.bbox is None]
    if no_bbox:
        print(f"  {name}: no reprojectable CRS, item has no geometry: {', '.join(no_bbox)}")
    bboxes = [i.bbox for i in items if i.bbox] or [[-180, -90, 180, 90]]
    times = sorted(i.datetime for i in items)
    collection = pystac.Collection(
        id=name,
        description=f"{name}: one Item per converted file ({', '.join(sorted({r['format'] for r in recs}))})",
        extent=pystac.Extent(
            spatial=pystac.SpatialExtent([[min(b[0] for b in bboxes), min(b[1] for b in bboxes),
                                           max(b[2] for b in bboxes), max(b[3] for b in bboxes)]]),
            temporal=pystac.TemporalExtent([[times[0], times[-1]]]),
        ),
        license="proprietary",
    )
    collection.add_items(items)
    return collection


def _folder_node(name, node):
    """Folder with only files -> Collection; folder with subfolders -> Catalog of one Collection per subfolder."""
    if not node["dirs"]:
        return _collection(name, _records(node))
    catalog = pystac.Catalog(id=name, description=f"{name}: one Collection per subfolder")
    for d, child in node["dirs"].items():
        catalog.add_child(_collection(d, _records(child)))
    for item in _items([r for rs in node["files"].values() for r in rs]):
        catalog.add_item(item)
    return catalog


def build_stac_catalog(results):
    """OUT_DIR/stac/<root>: STAC of the outputs of this run only (`results` from run_pipeline)."""
    results = results.astype(object).where(results.notna(), None)    # pandas turns missing crs / bounds into NaN
    recs = [r for r in results.to_dict("records") if r["status"] == "ok" and Path(r["output"]).exists()]
    if not recs:
        print("nothing converted in this run, no STAC written")
        return None
    root = _collapse_archives(_tree(recs))
    if len(root["dirs"]) == 1 and not root["files"]:             # one input folder: it is the root
        (name, node), = root["dirs"].items()
        stac = _folder_node(name, node)
    else:                                                          # several inputs / loose files: a run root
        name = f"run_{datetime.now():%Y%m%dT%H%M%S}"
        if not root["dirs"]:
            stac = _collection(name, _records(root))
        else:
            stac = pystac.Catalog(id=name, description=f"Conversion run {name}")
            used = set()
            for d, child in root["dirs"].items():
                stac.add_child(_folder_node(_unique(d, used), child))
            for item in _items([r for rs in root["files"].values() for r in rs]):
                stac.add_item(item)
    out = STAC_DIR / stac.id
    shutil.rmtree(out, ignore_errors=True)    # pystac never deletes Items that are no longer in the catalog
    stac.normalize_hrefs(str(out))
    stac.make_all_asset_hrefs_relative()
    stac.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
    _print_stac(stac)
    root_file = Path(stac.self_href).relative_to(OUT_DIR).as_posix()
    print("\nAsset hrefs are relative (../../../raster/...), so serve OUT_DIR itself, not just the stac folder:\n"
          f'  npx http-server "{OUT_DIR}" -p 8000 --cors -c-1\n'
          f"  SB_catalogUrl=http://localhost:8000/{root_file}")
    return stac


def _print_stac(node, indent=0):
    kind = "Collection" if isinstance(node, pystac.Collection) else "Catalog"
    items = list(node.get_items())
    print(f"{'  ' * indent}{kind} {node.id}" + (f": {len(items)} items" if items else ""))
    for child in node.get_children():
        _print_stac(child, indent + 1)


stac_catalog = build_stac_catalog(results)

  run_20261021T141020: no reprojectable CRS, item has no geometry: lc, chiba_city
Collection run_20261021T141020: 28 items

Asset hrefs are relative (../../../raster/...), so serve OUT_DIR itself, not just the stac folder:
  npx http-server "C:\Users\Pedro\Documents\PSS\data\cng" -p 8000 --cors -c-1
  SB_catalogUrl=http://localhost:8000/stac/run_20261021T141020/collection.json
